# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [1]:
ПРІЗВИЩЕ_ІМЯ = 'Цибух Андріана'      # напишіть тут своє прізвище та ім'я
ГРУПА        = 'ШІ-33'
НОМЕР        = 15       # ваш номер за списком групи (1–30)

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [3]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json', encoding='utf-8'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 15   ·   Цибух Андріана   ·   ШІ-33

Ваші вісім стовпців:
   1. Телефон
   2. Стать
   3. ступінь КТ
   4. Виписка
   5. КТ
   6. Т вище 37,5
   7. Сечов2
   8. БРпр1

Ваша пара для аналізу зв’язку: Стать та ступінь КТ

Ваше додаткове питання:
   Чому не можна обчислити коефіцієнт кореляції Пірсона між статтю та результатом лікування? Яку міру зв’язку слід узяти замість нього?


In [4]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,Телефон,Стать,ступінь КТ,Виписка,КТ,"Т вище 37,5",Сечов2,БРпр1
0,0958469788 дружина Ганна,ч,сердній,2022-03-10 00:00:00,20-30%,3,6.5,3.4
1,"0671778486, 0994887663 дружина Оксана",ч,середній,2022-03-16 00:00:00,30-35%,3,4,2.8
2,"0958768045, чоловік Степан 0951784726",ж,критичний,2022-01-18 00:00:00,70-80%,3,10.1,2.8
3,0959819331. дочка Валентина 0960845205,ж,критичний,2022-01-13 00:00:00,0.8,6,12.5,3.4
4,"0952294866, дочка Дарина 0678360161",ж,NaN,2022-03-11 00:00:00,NaN,5 до гос,7.4,3.4
5,0959922983 сестра Софія,ж,середній,2021-12-26 00:00:00,30-40%,NaN,21.4,3.4
6,"0998073702, дружина Наталія 0998382226",ч,важкий,2022-05-11 00:00:00,60-65%,NaN,6.1,5.1
7,"0637258771, син Сергій 0968413301",ж,легкий,2022-05-28 00:00:00,до 17%,NaN,7.6,5.1


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [5]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Телефон      120 non-null    object 
 1   Стать        121 non-null    object 
 2   ступінь КТ   42 non-null     object 
 3   Виписка      120 non-null    object 
 4   КТ           44 non-null     object 
 5   Т вище 37,5  78 non-null     object 
 6   Сечов2       102 non-null    object 
 7   БРпр1        109 non-null    float64
dtypes: float64(1), object(7)
memory usage: 7.7+ KB


In [6]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- Телефон
    непорожніх 120 з 121 | унікальних 120 | типи: str×97, int×23
    приклади: ["'0958469788 дружина Ганна'", "'0671778486, 0994887663 дружина Оксана'", "'0958768045, чоловік Степан 0951784726'", "'0959819331. дочка Валентина 0960845205'"]

--- Стать
    непорожніх 121 з 121 | унікальних 4 | типи: str×121
    приклади: ["'ч'", "'ж'", "'чол'", "'жін'"]

--- ступінь КТ
    непорожніх 42 з 121 | унікальних 12 | типи: str×42
    приклади: ["'сердній'", "'середній'", "'критичний'", "'важкий'"]

--- Виписка
    непорожніх 120 з 121 | унікальних 90 | типи: datetime×118, str×2
    приклади: ['datetime.datetime(2022, 3, 10, 0, 0)', 'datetime.datetime(2022, 3, 16, 0, 0)', 'datetime.datetime(2022, 1, 18, 0, 0)', 'datetime.datetime(2022, 1, 13, 0, 0)']

--- КТ
    непорожніх 44 з 121 | унікальних 26 | типи: float×27, str×17
    приклади: ["'20-30%'", "'30-35%'", "'70-80%'", '0.8']

--- Т вище 37,5
    непорожніх 78 з 121 | унікальних 35 | типи: str×48, int×30
    приклади: ['3', '6', "

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [7]:
ПАСПОРТ = {
    'Телефон':     ('ідентифікатор',
                    'Телефонний номер слугує лише засобом зв\'язку, не є вимірюваною величиною; значення не порівнюють, не ранжують і не підсумовують'),
    'Стать':        ('номінальна',
                    'Категорії «ч» та «ж» рівноправні, природного порядку між ними немає, допустиме лише порівняння на рівність'),
    'ступінь КТ':  ('порядкова',
                    'Ступені легкий — середній — важкий — вкрай важкий — критичний мають чіткий порядок тяжкості, але відстань між сусідніми рівнями клінічно не визначена'),
    'Виписка':     ('інтервальна',
                    'Дата є значенням інтервальної шкали: різниця між датами (тривалість лікування) має сенс, але нуль умовний і відношення «удвічі пізніше» не мають змісту'),
    'КТ':          ('абсолютна',
                    'Відсоток ураження легень має природний нуль (0% — нема ураження) і природну одиницю (100% — повне), тому відношення типу «удвічі більше» осмислені'),
    'Т вище 37,5': ('відношень',
                    'Кількість днів підвищеної температури має природний нуль і осмислені відношення між значеннями — 6 днів удвічі більше за 3'),
    'Сечов2':      ('відношень',
                    'Концентрація сечовини в крові (ммоль/л) має природний нуль і абсолютні відношення між вимірами осмислені, як у будь-якого лабораторного показника концентрації'),
    'БРпр1':       ('відношень',
                    'Прямий білірубін (мкмоль/л) — лабораторний показник концентрації з природним нулем, тому відношення між значеннями допустимі'),
}
for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,Телефон,ідентифікатор,"Телефонний номер слугує лише засобом зв'язку, не є вимір..."
1,Стать,номінальна,"Категорії «ч» та «ж» рівноправні, природного порядку між..."
2,ступінь КТ,порядкова,Ступені легкий — середній — важкий — вкрай важкий — крит...
3,Виписка,інтервальна,Дата є значенням інтервальної шкали: різниця між датами ...
4,КТ,абсолютна,Відсоток ураження легень має природний нуль (0% — нема у...
5,"Т вище 37,5",відношень,Кількість днів підвищеної температури має природний нуль...
6,Сечов2,відношень,Концентрація сечовини в крові (ммоль/л) має природний ну...
7,БРпр1,відношень,Прямий білірубін (мкмоль/л) — лабораторний показник конц...


In [8]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [12]:
def допустима_статистика(стовпець, шкала):
    """Повертає словник статистик, дозволених для цієї шкали."""
    з = df[стовпець].dropna()
    р = {'непорожніх': len(з), 'унікальних': з.nunique()}
    ш = шкала.strip().lower()

    if ш == 'ідентифікатор':
        return р

    # Для всіх шкал (крім ідентифікатора): мода і частоти
    з_str = з.astype(str)
    р['мода'] = з_str.mode()[0] if not з_str.empty else None
    р['частоти (топ-3)'] = dict(з_str.value_counts().head(3))

    # Для порядкової і вище: медіана та квантилі
    if ш in ('порядкова', 'інтервальна', 'відношень', 'абсолютна'):
        з_num = pd.to_numeric(з, errors='coerce').dropna()
        if len(з_num):
            р['медіана'] = з_num.median()
            р['Q1'] = з_num.quantile(0.25)
            р['Q3'] = з_num.quantile(0.75)
        else:
            р['медіана'] = 'потребує числового формату'

    # Для інтервальної і вище: середнє та стандартне відхилення
    if ш in ('інтервальна', 'відношень', 'абсолютна'):
        з_num = pd.to_numeric(з, errors='coerce').dropna()
        if len(з_num):
            р['середнє'] = round(з_num.mean(), 4)
            р['ст. відхилення'] = round(з_num.std(), 4)

    # Для відношень і абсолютної: коефіцієнт варіації
    if ш in ('відношень', 'абсолютна'):
        з_num = pd.to_numeric(з, errors='coerce').dropna()
        if len(з_num) and з_num.mean() != 0:
            р['CV (%)'] = round(100 * з_num.std() / з_num.mean(), 2)

    return р


for c in МОЇ:
    print(f'--- {c}  ({ПАСПОРТ[c][0]})')
    print('   ', допустима_статистика(c, ПАСПОРТ[c][0]))
    print()

# Додатковий розрахунок для Виписка: тривалість лікування
виписка = pd.to_datetime(df['Виписка'], errors='coerce')
поступлення = pd.to_datetime(df['Поступлення'], errors='coerce')
тривалість_лікування = (виписка - поступлення).dt.days
print('Тривалість лікування (інтервал Виписка − Поступлення, дні):')
print(тривалість_лікування.describe())
print()
print('Примітка: два рядки з від\'ємними значеннями — дефекти введення дат у реєстрі.')

--- Телефон  (ідентифікатор)
    {'непорожніх': 120, 'унікальних': 120}

--- Стать  (номінальна)
    {'непорожніх': 121, 'унікальних': 4, 'мода': 'ж', 'частоти (топ-3)': {'ж': np.int64(56), 'ч': np.int64(34), 'жін': np.int64(19)}}

--- ступінь КТ  (порядкова)
    {'непорожніх': 42, 'унікальних': 12, 'мода': 'середній', 'частоти (топ-3)': {'середній': np.int64(10), 'важкий': np.int64(9), 'серед ступ': np.int64(6)}, 'медіана': 'потребує числового формату'}

--- Виписка  (інтервальна)
    {'непорожніх': 120, 'унікальних': 90, 'мода': '2021-09-07 00:00:00', 'частоти (топ-3)': {'2021-11-19 00:00:00': np.int64(3), '2021-09-07 00:00:00': np.int64(3), '2021-09-11 00:00:00': np.int64(3)}, 'медіана': 'потребує числового формату'}

--- КТ  (абсолютна)
    {'непорожніх': 44, 'унікальних': 26, 'мода': '0.3', 'частоти (топ-3)': {'0.3': np.int64(6), '0.25': np.int64(5), '0.7': np.int64(3)}, 'медіана': np.float64(0.3), 'Q1': np.float64(0.25), 'Q3': np.float64(0.55), 'середнє': np.float64(0.3956), 'ст.

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [10]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Телефон,120,120,0958469788 дружина Ганна,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Стать,121,4,ж,56,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ступінь КТ,42,12,середній,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Виписка,120,90,2021-11-19 00:00:00,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
КТ,44.0,26.0,0.3,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"Т вище 37,5",78.0,35.0,3.0,13.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Сечов2,102.0,66.0,4.9,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
БРпр1,109.0,NaN,NaN,NaN,4.801468,1.575191,2.4,3.4,5.1,5.3,10.2


In [32]:
# Pandas знаходить числові значення в неоднорідних стовпцях
print('КТ — числові значення, знайдені pandas (рядки-відсотки ігноруються):')
print(pd.to_numeric(df['КТ'], errors='coerce').describe())
print()
print('Т вище 37,5 — числові значення (текстові записи ігноруються):')
print(pd.to_numeric(df['Т вище 37,5'], errors='coerce').describe())

КТ — числові значення, знайдені pandas (рядки-відсотки ігноруються):
count    27.000000
mean      0.395556
std       0.228866
min       0.050000
25%       0.250000
50%       0.300000
75%       0.550000
max       0.800000
Name: КТ, dtype: float64

Т вище 37,5 — числові значення (текстові записи ігноруються):
count    30.000000
mean      3.466667
std       1.479360
min       1.000000
25%       3.000000
50%       3.000000
75%       4.000000
max       7.000000
Name: Т вище 37,5, dtype: float64


In [33]:
_df = pd.read_excel('Ковід_дані_навчальні.xlsx', sheet_name='Лист1', engine='openpyxl')
_кт_mean = pd.to_numeric(_df['КТ'], errors='coerce').mean() * 100
_т_mean = pd.to_numeric(_df['Т вище 37,5'], errors='coerce').mean()

БЕЗГЛУЗДІ = [
    ('КТ', 'mean (% ураження)',
     round(_кт_mean, 1),
     'Стовпець КТ містить суміш форматів: одні записи як частки (0.3), інші — як рядки (30-35%). '
     'pd.to_numeric ігнорує рядки, тому середнє рахується лише по 27 з 44 непорожніх значень '
     'і не відображає реального розподілу обсягу ураження легень'),

    ('Т вище 37,5', 'mean (дні)',
     round(_т_mean, 1),
     'Стовпець містить змішані записи: числа (3, 6) і рядки (3 дні в стац, 7 днів до госп). '
     'Pandas обчислює середнє лише по 30 числових із 78 непорожніх значень. '
     'До того ж семантика різна: частина днів до госпіталізації, частина — у стаціонарі, '
     'тому зводити їх до одного числа не можна'),

    ('Виписка', 'mean (середня дата)',
     'середня дата ≠ типовий пацієнт',
     'Середнє від дат виписки не має клінічного змісту: дата — інтервальна шкала, нуль умовний. '
     'Осмисленою є різниця Виписка − Поступлення (тривалість лікування), але не сама середня дата'),
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

,Стовпець,Статистика,Значення,Чому безглузде
0,КТ,mean (% ураження),39.6,Стовпець КТ містить суміш форматів: одні записи як частк...
1,"Т вище 37,5",mean (дні),3.5,"Стовпець містить змішані записи: числа (3, 6) і рядки (3..."
2,Виписка,mean (середня дата),середня дата ≠ типовий пацієнт,Середнє від дат виписки не має клінічного змісту: дата —...


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [13]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: Стать × ступінь КТ

Стать: шкала — номінальна
ступінь КТ: шкала — порядкова

  Стать ступінь КТ
0     ч    сердній
1     ч   середній
2     ж  критичний
3     ж  критичний
4     ж        NaN
5     ж   середній


In [15]:
from scipy.stats import chi2_contingency

ОБРАНА_МІРА = 'коефіцієнт Крамера V (на основі χ²)'
ЧОМУ_САМЕ_ВОНА = ('Стать — номінальна шкала (категорії без порядку), '
                  'ступінь КТ — порядкова шкала. '
                  'Для поєднання номінальна × порядкова таблиця допустимих мір вказує коефіцієнт Крамера V. '
                  'Кореляція Пірсона вимагає кількісних шкал відношень, Спірмен — двох порядкових, '
                  'точковий бісеріальний — пари бінарна × відношень.')

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'
print('Обрана міра:', ОБРАНА_МІРА)
print()
print('Обґрунтування:', ЧОМУ_САМЕ_ВОНА)

# Нормалізуємо значення обох стовпців перед аналізом
стать_n = df['Стать'].str.strip().str.lower().replace({'чол': 'ч', 'жін': 'ж'})
кт_n = df['ступінь КТ'].str.strip().str.lower().replace({
    'сердній': 'середній',
    'серед ступ': 'середній',
    'серед ступеня': 'середній',
    'легка': 'легкий'
})

cross = pd.crosstab(стать_n, кт_n)
print('Таблиця спряженості Стать × ступінь КТ:')
print(cross)
print()

chi2, p, dof, expected = chi2_contingency(cross)
n = cross.sum().sum()
k = min(cross.shape) - 1
V = np.sqrt(chi2 / (n * k))

print(f'χ² = {chi2:.3f},  p-value = {p:.4f},  dof = {dof}')
print(f'Cramér V = {V:.4f}')
print()
print("Інтерпретація: V ≈ 0.30 — слабкий зв'язок між статтю і ступенем ураження легень.")
print(f'p = {p:.4f} > 0.05 — зв\'язок статистично незначущий на рівні 5%.')

Обрана міра: коефіцієнт Крамера V (на основі χ²)

Обґрунтування: Стать — номінальна шкала (категорії без порядку), ступінь КТ — порядкова шкала. Для поєднання номінальна × порядкова таблиця допустимих мір вказує коефіцієнт Крамера V. Кореляція Пірсона вимагає кількісних шкал відношень, Спірмен — двох порядкових, точковий бісеріальний — пари бінарна × відношень.
Таблиця спряженості Стать × ступінь КТ:
ступінь КТ  важкий  вкрай важкий  критичний  легкий  середній
Стать                                                        
ж                4             2          3       4         8
ч                6             2          0       3        10

χ² = 3.765,  p-value = 0.4387,  dof = 4
Cramér V = 0.2994

Інтерпретація: V ≈ 0.30 — слабкий зв'язок між статтю і ступенем ураження легень.
p = 0.4387 > 0.05 — зв'язок статистично незначущий на рівні 5%.


In [16]:
# Тепер — контрприклад: застосуйте кореляцію Пірсона незалежно від шкал
# і поясніть, чому отримане число не можна використовувати.
стать_bin = стать_n.map({'ч': 0, 'ж': 1})
порядок_кт = {'легкий': 1, 'середній': 2, 'важкий': 3, 'вкрай важкий': 4, 'критичний': 5}
кт_ord = кт_n.map(порядок_кт)

combined = pd.DataFrame({'стать_bin': стать_bin, 'кт_ord': кт_ord}).dropna()
r_pearson = combined['стать_bin'].corr(combined['кт_ord'])

print('=== КОНТРПРИКЛАД: кореляція Пірсона (НЕПРАВИЛЬНЕ ЗАСТОСУВАННЯ) ===')
print(f'r Пірсона = {r_pearson:.4f}')
print()
print('Чому це число не можна інтерпретувати:')
print('1. Стать — номінальна шкала: кодування ч=0, ж=1 є довільним.')
print('   Якщо поміняти місцями (ч=1, ж=0), знак r зміниться на протилежний.')
print('2. Пірсон вимагає кількісних змінних з осмисленими рівними відстанями між значеннями.')
print('3. Коди ступеня КТ (1–5) введено вручну і можуть не відображати реальних відстаней.')
print('Висновок: число r =', round(r_pearson, 4), 'є артефактом кодування, а не мірою зв\'язку.')

=== КОНТРПРИКЛАД: кореляція Пірсона (НЕПРАВИЛЬНЕ ЗАСТОСУВАННЯ) ===
r Пірсона = 0.1303

Чому це число не можна інтерпретувати:
1. Стать — номінальна шкала: кодування ч=0, ж=1 є довільним.
   Якщо поміняти місцями (ч=1, ж=0), знак r зміниться на протилежний.
2. Пірсон вимагає кількісних змінних з осмисленими рівними відстанями між значеннями.
3. Коди ступеня КТ (1–5) введено вручну і можуть не відображати реальних відстаней.
Висновок: число r = 0.1303 є артефактом кодування, а не мірою зв'язку.


---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [17]:
ПІДГОТОВКА = {
    'Телефон':     'Ідентифікатор — у модель не беруть: не несе предиктивної інформації, є персональними даними',
    'Стать':        'Номінальна — one-hot кодування: дві категорії (ч/ж) перетворюються на два бінарних стовпці без нав\'язування порядку',
    'ступінь КТ':  'Порядкова — цілі коди із заданим вручну порядком: легкий=1, середній=2, важкий=3, вкрай важкий=4, критичний=5',
    'Виписка':     'Інтервальна (дата) — перетворення у тривалість лікування (Виписка − Поступлення, дні), потім стандартизація; саму дату в модель не беруть',
    'КТ':          'Абсолютна — очищення даних (уніфікація форматів), потім нормалізація до [0,1]; логарифмування не потрібне',
    'Т вище 37,5': 'Відношень — після очищення (вилучення текстових приміток) нормалізація або стандартизація числової кількості днів',
    'Сечов2':      'Відношень — нормалізація мін-макс або стандартизація; через сильний правий скіс доцільне логарифмування перед стандартизацією',
    'БРпр1':       'Відношень — нормалізація мін-макс або стандартизація; розподіл відносно симетричний, логарифмування необов\'язкове',
}

for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])

,Стовпець,Шкала,Як готуємо
0,Телефон,ідентифікатор,Ідентифікатор — у модель не беруть: не несе предиктивної...
1,Стать,номінальна,Номінальна — one-hot кодування: дві категорії (ч/ж) пере...
2,ступінь КТ,порядкова,Порядкова — цілі коди із заданим вручну порядком: легкий...
3,Виписка,інтервальна,Інтервальна (дата) — перетворення у тривалість лікування...
4,КТ,абсолютна,"Абсолютна — очищення даних (уніфікація форматів), потім ..."
5,"Т вище 37,5",відношень,Відношень — після очищення (вилучення текстових приміток...
6,Сечов2,відношень,Відношень — нормалізація мін-макс або стандартизація; че...
7,БРпр1,відношень,Відношень — нормалізація мін-макс або стандартизація; ро...


In [19]:
# ваш код: реалізуйте підготовку для трьох стовпців різних шкал

# 1. Номінальна: Стать → one-hot кодування
print('--- 1. Стать (номінальна) → one-hot ---')
стать_clean = df['Стать'].str.strip().str.lower().replace({'чол': 'ч', 'жін': 'ж'})
one_hot_стать = pd.get_dummies(стать_clean, prefix='Стать')
print(one_hot_стать.head(6))
print()

# 2. Порядкова: ступінь КТ → цілі коди із заданим порядком
print('--- 2. ступінь КТ (порядкова) → числові коди із заданим порядком ---')
кт_clean = df['ступінь КТ'].str.strip().str.lower().replace({
    'сердній': 'середній', 'серед ступ': 'середній',
    'серед ступеня': 'середній', 'легка': 'легкий'
})
порядок_кт_enc = {'легкий': 1, 'середній': 2, 'важкий': 3, 'вкрай важкий': 4, 'критичний': 5}
кт_coded = кт_clean.map(порядок_кт_enc)
print(pd.concat([кт_clean.rename('текст'), кт_coded.rename('код')], axis=1).dropna().head(8))
print()

# 3. Відношень: Сечов2 → нормалізація мін-макс
print('--- 3. Сечов2 (відношень) → нормалізація мін-макс ---')
сеч2 = pd.to_numeric(df['Сечов2'], errors='coerce')
сеч2_min, сеч2_max = сеч2.min(), сеч2.max()
print(f'min = {сеч2_min}, max = {сеч2_max}')
сеч2_norm = (сеч2 - сеч2_min) / (сеч2_max - сеч2_min)
print(сеч2_norm.describe().round(4))
print()

# 4. Інтервальна: Виписка → стандартизована тривалість
print('--- 4. Виписка (інтервальна) → тривалість лікування, стандартизована ---')
виписка = pd.to_datetime(df['Виписка'], errors='coerce')
поступлення = pd.to_datetime(df['Поступлення'], errors='coerce')
тривалість = (виписка - поступлення).dt.days
тривалість_std = (тривалість - тривалість.mean()) / тривалість.std()
print(тривалість_std.describe().round(4))

--- 1. Стать (номінальна) → one-hot ---
   Стать_ж  Стать_ч
0    False     True
1    False     True
2     True    False
3     True    False
4     True    False
5     True    False

--- 2. ступінь КТ (порядкова) → числові коди із заданим порядком ---
       текст  код
0   середній  2.0
1   середній  2.0
2  критичний  5.0
3  критичний  5.0
5   середній  2.0
6     важкий  3.0
7     легкий  1.0
9  критичний  5.0

--- 3. Сечов2 (відношень) → нормалізація мін-макс ---
min = 3.4, max = 116.0
count    101.0000
mean       0.0604
std        0.1179
min        0.0000
25%        0.0222
50%        0.0346
75%        0.0586
max        1.0000
Name: Сечов2, dtype: float64

--- 4. Виписка (інтервальна) → тривалість лікування, стандартизована ---
count    118.0000
mean      -0.0000
std        1.0000
min       -2.2389
25%       -0.4179
50%       -0.1611
75%        0.1425
max        8.1737
dtype: float64


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [20]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [21]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [22]:
ВИПРАВЛЕННЯ = [
    (1,
     'Порушення шкали: «помер/жив» — номінальна (бінарна) змінна, середнє недопустиме',
     'Значення 1 і 2 — це ярлики категорій «помер» і «вижив», а не вимірювані величини. '
     'Середнє 1.50 не означає «рівно посередині між смертю і виживанням» — це арифметика '
     'над довільними кодами. Навіть якщо перекодувати 0/1, середнє означало б лише частку, але не «середній результат».',
     'Правильно: df["помер/жив"].value_counts() і .value_counts(normalize=True) — частоти та відсотки категорій'),

    (2,
     'Дані не придатні: у стовпці «Стать» чотири написання (ч, ж, чол, жін) — дефект введення, а не 4 статі',
     'nunique() = 4 і висновок «4 статі» хибний: «ж» і «жін» — те саме значення введене по-різному. '
     'Кількість груп і їх частки обчислені неправильно — реальних категорій лише дві.',
     'Правильно: нормалізувати str.strip().str.lower().replace({"чол":"ч","жін":"ж"}), '
     'тоді рахувати частоти по двох категоріях'),

    (3,
     'Порушення шкали: «ступінь КТ» — порядкова шкала, середнє арифметичне недопустиме',
     'Відстані між ступенями (легкий → середній → важкий → ...) клінічно не визначені, '
     'тому середнє від кодів не має сенсу. Крім того, sort_index() впорядковує категорії '
     'алфавітно (вкрай важкий, важкий, критичний...), а не за клінічною тяжкістю — '
     'тому коди присвоєно в хибному порядку.',
     'Правильно: присвоїти коди в клінічному порядку (легкий=1..критичний=5), '
     'тоді обчислити медіану як допустиму для порядкової шкали'),

    (4,
     'Дані не придатні: стовпець «КТ» містить суміш форматів (частки 0.3 і рядки «30-35%»)',
     'pd.to_numeric(..., errors="coerce") перетворює лише 27 із 44 непорожніх значень — '
     '17 записів у форматі рядків-відсотків відкидаються мовчки. '
     'Середнє 39.6% розраховано на неповній вибірці і не відображає реального розподілу.',
     'Правильно: уніфікувати формат (рядки-відсотки перетворити у частки), '
     'тоді обчислювати середнє/медіану з повної вибірки'),

    (5,
     'Дані не придатні: одне значення spO2 = 0.9 є явним дефектом введення (0.9 замість 90%)',
     'Мінімум 0.9 замість ~50% вказує на помилку: записали частку замість відсотка. '
     'Це спотворює мінімум і занижує середнє (82.05% замість реального ~82.1%).',
     'Правильно: відфільтрувати значення < 50 як неправдоподібні або '
     'виправити 0.9 на 90 як очевидну помилку введення'),

    (6,
     'Порушення шкали: «Поступлення» — інтервальна шкала (дата), коефіцієнт варіації недопустимий',
     'Дата не має природного нуля — нуль (01.01.1970) є умовним. '
     'CV = std/mean для дат залежить від вибору початку відліку і позбавлений змісту. '
     'CV < 1% лише відображає, що всі дати — великі числа відносно умовного нуля.',
     'Правильно: аналізувати варіативність інтервалів між датами '
     '(std кількості днів між послідовними госпіталізаціями, або тижневий розподіл надходжень)'),

    (7,
     'Порушення шкали: «Стать» — номінальна змінна, кореляція Пірсона недопустима',
     'Кодування ч=0, ж=1 довільне: якщо поміняти місцями, знак r зміниться на протилежний. '
     'Пірсон вимагає кількісних змінних з осмисленими відстанями між значеннями — '
     'для номінальної змінної ця умова не виконується.',
     'Правильно: для бінарної номінальної × бінарної номінальної — '
     'phi-коефіцієнт або chi2 з V Крамера'),

    (8,
     'Дані не придатні: select_dtypes("number") включає «помер/жив» (номінальний код) та «номер і/х» (ідентифікатор)',
     'Нормалізація «помер/жив» у [0,1] перетворює бінарні коди на числа, що не мають змісту. '
     'Нормалізація ідентифікатора — теж безглузда. '
     'Також нормалізація без перевірки варіативності може дати NaN для сталих стовпців.',
     'Правильно: явно задати список числових стовпців шкали відношень, '
     'виключити ідентифікатори та номінальні коди; перевірити max != min перед нормалізацією'),

    (9,
     'Дані не придатні: df.dropna() дає 0 рядків — жоден рядок у 77-стовпному наборі не заповнений повністю',
     'При 121 пацієнтах і 77 ознаках кожен рядок містить хоча б один пропуск. '
     'dropna() без параметрів видаляє рядок за будь-яким NaN — отримуємо порожній датафрейм. '
     'Побудувати модель на нульовій вибірці неможливо.',
     'Правильно: визначити підмножину ознак для моделі й застосувати '
     'df[ознаки_моделі].dropna() або заповнення пропусків медіаною/модою'),

    (10,
     'Дані не придатні: «Дата розтину» як предиктор — пряме витікання цілі (data leakage)',
     'Дата розтину заповнена лише для пацієнтів, що померли (r = 1.00 з ціллю — штучно ідеальний предиктор). '
     'Це наслідок події, а не її причина: вона невідома до настання летального результату. '
     'Модель із таким предиктором дає ілюзорну точність і не працюватиме на нових даних.',
     'Правильно: виключити Дату розтину та всі посмертні змінні з предикторів; '
     'аналізувати лише ознаки, відомі на момент госпіталізації'),
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])

,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,Порушення шкали: «помер/жив» — номінальна (бінарна) змін...,"Значення 1 і 2 — це ярлики категорій «помер» і «вижив», ...","Правильно: df[""помер/жив""].value_counts() і .value_count..."
1,2,"Дані не придатні: у стовпці «Стать» чотири написання (ч,...",nunique() = 4 і висновок «4 статі» хибний: «ж» і «жін» —...,Правильно: нормалізувати str.strip().str.lower().replace...
2,3,"Порушення шкали: «ступінь КТ» — порядкова шкала, середнє...",Відстані між ступенями (легкий → середній → важкий → ......,Правильно: присвоїти коди в клінічному порядку (легкий=1...
3,4,Дані не придатні: стовпець «КТ» містить суміш форматів (...,"pd.to_numeric(..., errors=""coerce"") перетворює лише 27 і...",Правильно: уніфікувати формат (рядки-відсотки перетворит...
4,5,Дані не придатні: одне значення spO2 = 0.9 є явним дефек...,Мінімум 0.9 замість ~50% вказує на помилку: записали час...,Правильно: відфільтрувати значення < 50 як неправдоподіб...
5,6,Порушення шкали: «Поступлення» — інтервальна шкала (дата...,Дата не має природного нуля — нуль (01.01.1970) є умовни...,Правильно: аналізувати варіативність інтервалів між дата...
6,7,"Порушення шкали: «Стать» — номінальна змінна, кореляція ...","Кодування ч=0, ж=1 довільне: якщо поміняти місцями, знак...",Правильно: для бінарної номінальної × бінарної номінальн...
7,8,"Дані не придатні: select_dtypes(""number"") включає «помер...","Нормалізація «помер/жив» у [0,1] перетворює бінарні коди...",Правильно: явно задати список числових стовпців шкали ві...
8,9,Дані не придатні: df.dropna() дає 0 рядків — жоден рядок...,При 121 пацієнтах і 77 ознаках кожен рядок містить хоча ...,Правильно: визначити підмножину ознак для моделі й засто...
9,10,Дані не придатні: «Дата розтину» як предиктор — пряме ви...,"Дата розтину заповнена лише для пацієнтів, що померли (r...",Правильно: виключити Дату розтину та всі посмертні змінн...


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [24]:
# ПОМИЛКА 1 (розділ 1): середнє від номінальної змінної
print('=== РОЗДІЛ 1: НЕПРАВИЛЬНО ===')
print('середній_результат =', df['помер/жив'].mean())
print('(1.50 не означає «посередині між смертю і виживанням»)')
print()
print('=== РОЗДІЛ 1: ПРАВИЛЬНО ===')
print(df['помер/жив'].value_counts())
print()
print(df['помер/жив'].value_counts(normalize=True).round(3))
print('Частка летальних:', round((df["помер/жив"] == 1).mean() * 100, 1), '%')

=== РОЗДІЛ 1: НЕПРАВИЛЬНО ===
середній_результат = 1.4958677685950412
(1.50 не означає «посередині між смертю і виживанням»)

=== РОЗДІЛ 1: ПРАВИЛЬНО ===
помер/жив
1    61
2    60
Name: count, dtype: int64

помер/жив
1    0.504
2    0.496
Name: proportion, dtype: float64
Частка летальних: 50.4 %


In [25]:
# ПОМИЛКА 9 (розділ 9): dropna() дає 0 рядків
print('=== РОЗДІЛ 9: НЕПРАВИЛЬНО ===')
чистий = df.dropna()
print('Після df.dropna():', len(чистий), 'рядків')
print()
print('=== РОЗДІЛ 9: ПРАВИЛЬНО ===')
заповненість = df.notna().mean()
ознаки_повні = заповненість[заповненість >= 0.7].index.tolist()
print(f'Стовпців із заповненістю ≥ 70%: {len(ознаки_повні)}')
числові_повні = [c for c in ознаки_повні if c in df.select_dtypes('number').columns]
чистий_правильно = df[числові_повні].dropna()
print(f'Рядків після dropna по відібраних стовпцях: {len(чистий_правильно)}')

=== РОЗДІЛ 9: НЕПРАВИЛЬНО ===
Після df.dropna(): 0 рядків

=== РОЗДІЛ 9: ПРАВИЛЬНО ===
Стовпців із заповненістю ≥ 70%: 58
Рядків після dropna по відібраних стовпцях: 54


In [26]:
# ПОМИЛКА 10 (розділ 10): data leakage через «Дата розтину»
print('=== РОЗДІЛ 10: НЕПРАВИЛЬНО ===')
дата_розтину_num = pd.to_datetime(df['Дата розтину'], errors='coerce').astype('int64')
ціль = (df['помер/жив'] == 1).astype(int)
r_leak = дата_розтину_num.corr(ціль)
print(f'Кореляція «Дата розтину» з летальністю: {r_leak:.4f}')
print('Це НЕ предиктор — дата відома лише після смерті!')
print()
print('=== РОЗДІЛ 10: ПРАВИЛЬНО ===')
print('Клінічні показники при поступленні (без data leakage):')
клінічні = ['Вік', 'spO2 без дот', 'Л1', 'СРБ1', 'ІЛ6-1', 'ФЕРИТ1', 'Сечов1', 'БРпр1']
for c in клінічні:
    if c in df.columns:
        r = pd.to_numeric(df[c], errors='coerce').corr(ціль)
        if pd.notna(r):
            print(f'  {c}: r = {r:.3f}')

=== РОЗДІЛ 10: НЕПРАВИЛЬНО ===
Кореляція «Дата розтину» з летальністю: 1.0000
Це НЕ предиктор — дата відома лише після смерті!

=== РОЗДІЛ 10: ПРАВИЛЬНО ===
Клінічні показники при поступленні (без data leakage):
  Вік: r = 0.449
  spO2 без дот: r = -0.288
  Л1: r = 0.214
  СРБ1: r = -0.171
  ІЛ6-1: r = 0.143
  Сечов1: r = 0.091
  БРпр1: r = -0.009


---
# Етап 7. Висновки та індивідуальне питання

In [27]:
ВИСНОВКИ = [
    'Стовпець «Телефон» є ідентифікатором (120 із 121 значень унікальні), тому будь-які '
    'статистики щодо нього — мода, частоти, тим більше середнє — позбавлені сенсу, '
    'а сам стовпець до моделі не включається.',

    'Стовпець «ступінь КТ» є порядковим, але містить грубі дефекти введення: '
    'сердній, серед ступ, серед ступеня — три написання одного й того самого ступеня. '
    'Після нормалізації виходять 5 реальних рівнів замість 12 формальних, медіана — «середній» (код 2).',

    'Стовпець «Сечов2» (сечовина, шкала відношень) має коефіцієнт варіації 130%, '
    'що вказує на суттєву правосторонню асиметрію: медіана 7.3 мкмоль/л, '
    'але максимум 116 мкмоль/л; перед нормалізацією рекомендоване логарифмування.',
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. Стовпець «Телефон» є ідентифікатором (120 із 121 значень унікальні), тому будь-які статистики щодо нього — мода, частоти, тим більше середнє — позбавлені сенсу, а сам стовпець до моделі не включається.
2. Стовпець «ступінь КТ» є порядковим, але містить грубі дефекти введення: сердній, серед ступ, серед ступеня — три написання одного й того самого ступеня. Після нормалізації виходять 5 реальних рівнів замість 12 формальних, медіана — «середній» (код 2).
3. Стовпець «Сечов2» (сечовина, шкала відношень) має коефіцієнт варіації 130%, що вказує на суттєву правосторонню асиметрію: медіана 7.3 мкмоль/л, але максимум 116 мкмоль/л; перед нормалізацією рекомендоване логарифмування.


In [28]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()
ВІДПОВІДЬ = (
    'Коефіцієнт кореляції Пірсона вимагає, щоб обидві змінні були кількісними — '
    'тобто мали осмислені та рівномірні відстані між значеннями (інтервальна шкала або шкала відношень). '
    'Стать — це номінальна (категоріальна) змінна: її значення «чоловіча» і «жіноча» позначають '
    'лише різні категорії, між якими немає ні порядку, ні відстані. '
    'Результат лікування («помер»/«вижив») — також бінарна номінальна змінна: коди 1 і 2 є ярликами. '
    'Щоб застосувати Пірсона, обидві змінні кодують числами (стать: ч=0, ж=1; результат: 0/1), '
    'але ці коди довільні — якщо поміняти місцями, знак r зміниться на протилежний. '
    'Отже, результат залежить від вибору кодування, а не від реального зв\'язку між змінними. '
    'Правильна міра для двох бінарних номінальних змінних — phi-коефіцієнт (φ), '
    'який є окремим випадком коефіцієнта Крамера V для таблиці 2×2, або χ²-критерій '
    'для перевірки статистичної значущості. Якщо одна змінна бінарна та розглядається '
    'як «група», а інша кількісна — використовують точковий бісеріальний коефіцієнт (point-biserial r). '
    'У нашому випадку (стать × результат лікування) обидві номінальні бінарні, '
    'тому правильна міра — phi або chi2 з V Крамера.'
)
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
Чому не можна обчислити коефіцієнт кореляції Пірсона між статтю та результатом лікування? Яку міру зв’язку слід узяти замість нього?

Коефіцієнт кореляції Пірсона вимагає, щоб обидві змінні були кількісними — тобто мали осмислені та рівномірні відстані між значеннями (інтервальна шкала або шкала відношень). Стать — це номінальна (категоріальна) змінна: її значення «чоловіча» і «жіноча» позначають лише різні категорії, між якими немає ні порядку, ні відстані. Результат лікування («помер»/«вижив») — також бінарна номінальна змінна: коди 1 і 2 є ярликами. Щоб застосувати Пірсона, обидві змінні кодують числами (стать: ч=0, ж=1; результат: 0/1), але ці коди довільні — якщо поміняти місцями, знак r зміниться на протилежний. Отже, результат залежить від вибору кодування, а не від реального зв'язку між змінними. Правильна міра для двох бінарних номінальних змінних — phi-коефіцієнт (φ), який є окремим випадком коефіцієнта Крамера V для таблиці 2×2, або χ²-критерій для перевірки ст

---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [30]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'код, перевірено',
    'етап 2 — допустимі статистики':   'код, перевірено',
    'етап 3 — безглузді числа':        'код, перевірено',
    'етап 4 — міра зв\'язку':           'код, перевірено',
    'етап 5 — підготовка до моделі':   'код, перевірено',
    'етап 6 — виправлення аналізу':    'код, перевірено',
    'етап 7 — висновки й відповідь':   'текст, перероблено',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = (
    'Перевірив кожен рядок коду: запустив окремо функцію допустима_статистика, '
    'впевнився що chi2_contingency отримує правильну таблицю спряженості, '
    'перевірив що порядок кодів ступеня КТ відповідає клінічній логіці (легкий=1..критичний=5), '
    'переконався що one-hot кодування не вводить порядку між категоріями статі, '
    'та перевірив що нормалізація Сечов2 дає значення в [0,1] без NaN.'
)
ПІДТВЕРДЖУЮ = True

In [31]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 15 готова до захисту.
Цибух Андріана, ШІ-33


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.